In [1]:
!pip install easyocr


In [ ]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel, pipeline
import easyocr
import cv2
import numpy as np
import torchvision

def classify_clip(image, model, processor):
    labels = ["handwritten text", "printed text"]
    image = image.convert("RGB")
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(**inputs)
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1).squeeze()

    predicted = labels[probs.argmax()]
    return predicted, probs.max().item()

def extract_and_classify_text_regions(image_path):
    reader = easyocr.Reader(['ru', 'en'], gpu=True)
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Не удалось загрузить изображение")

    results = reader.readtext(image_path, detail=1)
    regions = []

    for i, (bbox, text, prob) in enumerate(results):
        (tl, tr, br, bl) = bbox
        x_min = int(min(tl[0], bl[0]))
        y_min = int(min(tl[1], tr[1]))
        x_max = int(max(tr[0], br[0]))
        y_max = int(max(bl[1], br[1]))

        text_region = img[y_min:y_max, x_min:x_max]
        if text_region.size == 0:
            continue

        text_region_pil = Image.fromarray(cv2.cvtColor(text_region, cv2.COLOR_BGR2RGB))
        predicted_label, confidence = classify_clip(text_region_pil, model, processor)

        regions.append({
            'image': text_region_pil,
            'bbox': (x_min, y_min, x_max, y_max),
            'label': predicted_label.replace(" text", ""),  # "handwritten" / "printed"
            'confidence': confidence
        })

    return regions

# Постобработка: NMS
def apply_nms(regions, iou_threshold=0.3):
    if len(regions) == 0:
        return []

    boxes = [list(region['bbox']) for region in regions]
    scores = [region['confidence'] for region in regions]

    boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
    scores_tensor = torch.tensor(scores)

    keep_indices = torchvision.ops.nms(boxes_tensor, scores_tensor, iou_threshold)
    filtered = [regions[i] for i in keep_indices]
    return filtered

def get_relevance_extracted_text(image_path):
    regions = extract_and_classify_text_regions(image_path)
    pipe = pipeline("image-to-text", model="raxtemur/trocr-base-ru", device=0)

    output_data = []
    for region in regions:
        image = region['image']
        result = pipe(image)

        output_data.append({
            "text": result[0]['generated_text'],
            "bbox": list(region['bbox']),
            "label": region['label'],
            "confidence": round(region['confidence'], 4)
        })

    final_json = {
        "results": output_data
    }

    return final_json

def visualize_text_blocks(image_path, regions, save_path="visualized_output.jpg"):
    img = cv2.imread(image_path)

    for region in regions:
        x_min, y_min, x_max, y_max = region['bbox']
        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

    cv2.imwrite(save_path, img)
    print(f"Визуализация сохранена в {save_path}")


d:\anaconda\envs\T1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\anaconda\envs\T1\lib\site-packages\transformers\utils\hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
import json

image_path = 'archive/jpg/jpg/a_1_1.jpg'

output_data = get_relevance_extracted_text(image_path)

visualize_text_blocks(image_path, output_data['results'], save_path="visualized_output.jpg")

output_json_path = 'output_data.json'
with open(output_json_path, 'w', encoding='utf-8') as json_file:
    json.dump(output_data, json_file, ensure_ascii=False, indent=4)

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "pooler_act": "tanh",
  "pooler_output_size": 768,
  "qkv_bias": false,
  "torch_dtype": "float32",
  "transformers_version": "4.51.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_mod

Визуализация сохранена в visualized_output.jpg


In [ ]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel, pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer_corr = AutoTokenizer.from_pretrained("google/flan-t5-small")
model_corr = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

def correct_text(text):
    prompt = f"Исправь ошибки в этом предложении, скорректируй его по смыслу: {text}"
    inputs = tokenizer_corr(prompt, return_tensors="pt")
    outputs = model_corr.generate(**inputs, max_new_tokens=100)
    return tokenizer_corr.decode(outputs[0], skip_special_tokens=True)

def classify_clip(image, model, processor):
    labels = ["handwritten text", "printed text"]
    image = image.convert("RGB")
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1).squeeze()

    predicted = labels[probs.argmax()]
    return predicted.replace(" text", ""), probs.max().item()

def extract_and_classify_text_regions(image_path):
    reader = easyocr.Reader(['ru', 'en'], gpu=True)
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Не удалось загрузить изображение")

    results = reader.readtext(image_path, detail=1)
    regions = []

    for bbox, text, prob in results:
        (top_left, top_right, bottom_right, bottom_left) = bbox
        x_min = int(min([top_left[0], bottom_left[0]]))
        y_min = int(min([top_left[1], top_right[1]]))
        x_max = int(max([top_right[0], bottom_right[0]]))
        y_max = int(max([bottom_left[1], bottom_right[1]]))

        text_region = img[y_min:y_max, x_min:x_max]
        if text_region.size == 0:
            continue

        text_region_pil = Image.fromarray(cv2.cvtColor(text_region, cv2.COLOR_BGR2RGB))
        predicted_label, confidence = classify_clip(text_region_pil, clip_model, clip_processor)

        regions.append({
            'image': text_region_pil,
            'bbox': (x_min, y_min, x_max, y_max),
            'label': predicted_label,
            'confidence': confidence
        })

    return regions

def visualize_text_regions(image_path, regions):
    img = cv2.imread(image_path)

    for region in regions:
        x_min, y_min, x_max, y_max = region['bbox']
        label = region['label']

        cv2.rectangle(img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)  

    cv2.imshow("Text Regions", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

def get_relevance_extracted_text(image_path, export_json=True, visualize=False):
    regions = extract_and_classify_text_regions(image_path)
    ocr_pipe = pipeline("image-to-text", model="raxtemur/trocr-base-ru", device=0)

    output_data = []

    for region in regions:
        image = region['image']
        ocr_result = ocr_pipe(image)
        raw_text = ocr_result[0]['generated_text']
        corrected_text = correct_text(raw_text)

        output_data.append({
            'text_raw': raw_text,
            'text_corrected': corrected_text,
            'bbox': region['bbox'],
            'label': region['label'],
            'confidence': region['confidence']
        })

    if export_json:
        with open("output_data.json", "w", encoding="utf-8") as f:
            json.dump(output_data, f, ensure_ascii=False, indent=2)

    if visualize:
        visualize_text_regions(image_path, regions)

    return output_data


tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
image_path = "a_1_1.jpg"

result = get_relevance_extracted_text(image_path, export_json=True, visualize=True)

import json
with open("output_data.json", "r", encoding="utf-8") as f:
    output_data = json.load(f)
    print(json.dumps(output_data, indent=2, ensure_ascii=False))
